In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from pathlib import Path

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [37]:
DATASET_DIR = Path("../datasets/raw/astro_dataset_maxia")

TRAIN_DIR = DATASET_DIR / "training"
VALID_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"

print(TRAIN_DIR)

..\datasets\raw\astro_dataset_maxia\training


In [52]:
DATASET_DIR = Path("../datasets/raw/astro_dataset_maxia")

print(DATASET_DIR.exists())
print(list(DATASET_DIR.iterdir()))

True
[WindowsPath('../datasets/raw/astro_dataset_maxia/test'), WindowsPath('../datasets/raw/astro_dataset_maxia/training'), WindowsPath('../datasets/raw/astro_dataset_maxia/validation')]


In [54]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=transform
)

valid_dataset = datasets.ImageFolder(
    VALID_DIR,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=transform
)

print(train_dataset.classes)
print(len(train_dataset))

['asteroid', 'black_hole', 'earth', 'galaxy', 'jupiter', 'mars', 'mercury', 'neptune', 'pluto', 'saturn', 'uranus', 'venus']
2416


In [96]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers = 4
)

In [82]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, len(train_dataset.classes))


    def forward(self, x):

        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)

        return x

In [97]:
print(type(train_dataset))
print(dir(train_dataset))

<class 'torchvision.datasets.folder.ImageFolder'>
['__add__', '__annotations__', '__class__', '__class_getitem__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__orig_bases__', '__parameters__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_format_transform_repr', '_repr_indent', 'class_to_idx', 'classes', 'extensions', 'extra_repr', 'find_classes', 'imgs', 'loader', 'make_dataset', 'root', 'samples', 'target_transform', 'targets', 'transform', 'transforms']


In [98]:
model = SimpleCNN().to(device)
print(model)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=100352, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=12, bias=True)
)


In [99]:
# loss function
criterion = nn.CrossEntropyLoss()
# optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print(criterion)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [100]:
# check one batch
images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [101]:
images = images.to(device)
outputs = model(images)

print(outputs.shape)

torch.Size([32, 12])


In [102]:
loss = criterion(outputs, labels)
print(loss)

tensor(2.5034, grad_fn=<NllLossBackward0>)


In [92]:
print("Training images:", len(train_dataset))
print("Training batches:", len(train_loader))

Training images: 2416
Training batches: 76


In [ ]:
# training loop prototype
num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        optimizer.step()
        
        running_loss += loss.item()
        
print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {running_loss / len(train_loader):.4f}")

Epoch [2/2] Loss: 0.0035


In [105]:
from tqdm.auto import tqdm
import time

# train the cnn
num_epochs = 10

# store loss history
train_losses = []

# total training time record
training_start = time.time()

for epoch in range(num_epochs):

    # start timing this epoch
    epoch_start = time.time()

    # put model into training mode
    model.train()

    # running loss
    running_loss = 0.0

    # progress bar
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{num_epochs}",
        leave=True
    )

    # loop over every batch
    for images, labels in progress_bar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # forward pass
        outputs = model(images)

        # compute loss
        loss = criterion(outputs, labels)

        # backpropagation
        loss.backward()

        # update weights
        optimizer.step()

        # add batch loss
        running_loss += loss.item()

        # update progress bar
        progress_bar.set_postfix(
            batch_loss=f"{loss.item():.4f}"
        )

    # average loss
    epoch_loss = running_loss / len(train_loader)

    train_losses.append(epoch_loss)

    epoch_time = time.time() - epoch_start

    print("-" * 50)
    print(f"Epoch {epoch+1}/{num_epochs} Complete")
    print(f"Average Loss : {epoch_loss:.4f}")
    print(f"Time Taken   : {epoch_time:.2f} sec")
    print("-" * 50)

# training Complete
total_time = time.time() - training_start

print("\nTraining Finished!")
print(f"Total Time: {total_time:.2f} seconds")


Epoch 1/10:   0%|          | 0/76 [00:20<?, ?it/s]

--------------------------------------------------
Epoch 1/10 Complete
Average Loss : 0.0014
Time Taken   : 80.81 sec
--------------------------------------------------


Epoch 2/10:   0%|          | 0/76 [00:21<?, ?it/s]

--------------------------------------------------
Epoch 2/10 Complete
Average Loss : 0.0009
Time Taken   : 100.97 sec
--------------------------------------------------


Epoch 3/10:   0%|          | 0/76 [00:20<?, ?it/s]

...

--------------------------------------------------
Epoch 8/10 Complete
Average Loss : 0.0003
Time Taken   : 154.92 sec
--------------------------------------------------


Epoch 9/10:   0%|          | 0/76 [00:19<?, ?it/s]

--------------------------------------------------
Epoch 9/10 Complete
Average Loss : 0.0003
Time Taken   : 75.29 sec
--------------------------------------------------


Epoch 10/10:   0%|          | 0/76 [00:21<?, ?it/s]

--------------------------------------------------
Epoch 10/10 Complete
Average Loss : 0.0003
Time Taken   : 134.79 sec
--------------------------------------------------

Training Finished!
Total Time: 1174.35 seconds


In [94]:
import os

print(os.cpu_count())

12


In [95]:
import time

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

start = time.time()

outputs = model(images)
loss = criterion(outputs, labels)
loss.backward()
optimizer.step()

print(time.time() - start)

0.4183931350708008


In [108]:
torch.save(model.state_dict(), "../models/simple_cnn.pth")
print("Model saved successfully!")

Model saved successfully!
